In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# 02 · Anomaly Detection — robust z-score + Prophet ensemble

Agent 2 (Anomaly Watchman) uses `backend.models.anomaly.AnomalyDetectionPipeline`:

* **Seasonal robust z-score** — expected = rolling median (+ hour-of-day profile when it helps); σ = 1.4826 × MAD.
* **Prophet** (optional) — trend + seasonality forecast; σ derived from the 99 % interval.

Both scores are in *robust sigmas*, so one severity scale applies: |score| ≥ 3.5 flagged, **low** < 4.5 ≤ **medium**
< 6 ≤ **high** < 9 ≤ **critical**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from backend.demo.synthetic import generate
from backend.models.anomaly import AnomalyDetectionPipeline, SEVERITY_ORDER, robust_baseline

## 1. Data
Same reproducible synthetic data as notebook 01. Timestamps are tz-aware (UTC).

In [ ]:
END = pd.Timestamp('2026-10-01 00:00', tz='UTC')
df = generate(periods=336, seed=42, end=END, incidents=True)
series = {m: g.set_index('timestamp')['value'].sort_index() for m, g in df.groupby('metric_name')}
print(len(series), 'series;', 'index tz =', series['daily_revenue'].index.tz)

## 2. Run the robust detector on every series
`use_prophet=False` keeps this fast and dependency-free.

In [ ]:
detector = AnomalyDetectionPipeline(z_threshold=3.5, use_prophet=False)

rows = []
for metric, s in series.items():
    for p in detector.detect(s):
        rows.append({'metric_name': metric, 'timestamp': p.timestamp, 'value': p.value,
                     'expected': p.expected, 'score': p.score, 'method': p.method, 'severity': p.severity})
anoms = pd.DataFrame(rows).sort_values(['timestamp', 'metric_name']).reset_index(drop=True)
print(len(anoms), 'anomalous points')
anoms.round({'value': 3, 'expected': 3, 'score': 3}).tail(20)

## 3. Count per severity

In [ ]:
sev_counts = anoms['severity'].value_counts().reindex(SEVERITY_ORDER, fill_value=0)
print(sev_counts.to_string())
sev_counts.plot.bar(color=['#9ecae1', '#fdae6b', '#e6550d', '#a50f15'], figsize=(5, 3), title='Anomalies by severity')
plt.ylabel('points')
plt.tight_layout()
plt.show()

In [ ]:
pd.crosstab(anoms['metric_name'], anoms['severity']).reindex(columns=SEVERITY_ORDER, fill_value=0)

## 4. Did we catch the injected incidents?
Ground truth from `backend/demo/synthetic.py`:
* **A** — `ad_spend_usd` collapses for 3 h starting 14 h before the end (then propagates down the marketing chain)
* **B** — `delivery_delay_hrs` spikes for 3 h starting 8 h before the end (then propagates down the operations chain)
* isolated point anomalies at fixed offsets from the end

In [ ]:
ts = series['daily_revenue'].index
injected = {
    'ad_spend_usd':          [ts[-14], ts[-13], ts[-12]],
    'delivery_delay_hrs':    [ts[-8], ts[-7], ts[-6]],
    'gross_margin_pct':      [ts[-30], ts[-200]],
    'avg_order_value':       [ts[-50]],
    'refund_count':          [ts[-20]],
    'cart_abandonment_rate': [ts[-90]],
    'inventory_turnover':    [ts[-120]],
}
flagged = set(zip(anoms['metric_name'], anoms['timestamp']))
check = pd.DataFrame([{'metric_name': m, 'timestamp': t, 'detected': (m, t) in flagged}
                      for m, times in injected.items() for t in times])
print(f"detected {check['detected'].sum()} / {len(check)} directly injected points")
check

Some injected points are small by design (≈ 4σ), so a miss there is expected. Propagated (downstream) anomalies are real effects of incidents A/B, not false positives.

## 5. Plot series with flagged points coloured by severity

In [ ]:
SEV_COLOR = {'low': '#3182bd', 'medium': '#fd8d3c', 'high': '#e6550d', 'critical': '#a50f15'}

def plot_metric(metric, ax):
    s = series[metric]
    expected, sigma = robust_baseline(s)
    ax.plot(s.index, s.values, lw=0.9, color='0.35', label='value')
    ax.plot(expected.index, expected.values, lw=0.8, color='tab:green', alpha=0.8, label='expected (robust)')
    ax.fill_between(expected.index, expected - 3.5 * sigma, expected + 3.5 * sigma, color='tab:green', alpha=0.12,
                    label='±3.5σ')
    a = anoms[anoms['metric_name'] == metric]
    for sev in SEVERITY_ORDER:
        pts = a[a['severity'] == sev]
        if len(pts):
            ax.scatter(pts['timestamp'], pts['value'], color=SEV_COLOR[sev], s=28, zorder=3, label=sev)
    ax.set_title(metric, fontsize=9)
    ax.legend(fontsize=7, loc='upper left', ncol=3)

metrics_to_plot = ['ad_spend_usd', 'conversion_rate', 'daily_revenue', 'delivery_delay_hrs', 'nps_score',
                   'gross_margin_pct']
fig, axes = plt.subplots(len(metrics_to_plot), 1, figsize=(12, 2.4 * len(metrics_to_plot)), sharex=True)
for ax, m in zip(axes, metrics_to_plot):
    plot_metric(m, ax)
plt.tight_layout()
plt.show()

## 6. Sensitivity to the threshold

In [ ]:
for thr in [3.0, 3.5, 4.5, 6.0]:
    d = AnomalyDetectionPipeline(z_threshold=thr, use_prophet=False)
    n = sum(len(d.detect(s)) for s in series.values())
    print(f'z_threshold={thr}: {n} flagged points')

## 7. Optional: add Prophet to the ensemble
When Prophet is installed and a series has ≥ 48 points, both detectors run; points flagged by both are reported as
`prophet+robust_z`. Prophet is slower, so we run it on two series only. If Prophet is not installed the pipeline
silently falls back to the robust detector.

In [ ]:
try:
    import prophet  # noqa: F401
    HAS_PROPHET = True
except Exception:
    HAS_PROPHET = False
print('Prophet available:', HAS_PROPHET)

ens = AnomalyDetectionPipeline(z_threshold=3.5, use_prophet=True)
for metric in ['ad_spend_usd', 'delivery_delay_hrs']:
    pts = ens.detect(series[metric])
    methods = pd.Series([p.method for p in pts]).value_counts().to_dict()
    print(f'{metric}: {len(pts)} points, by method: {methods}')

**Takeaway:** the robust detector flags both incidents and their downstream effects, and the shared robust-sigma scale lets Agent 2 gate alerts by severity regardless of which detector fired.